# AB Testi ile Bidding Yöntemlerinin Dönüşümünün Karşılaştırılması

## İş Problemi

Facebook kısa süre önce mevcut "maximum bidding" adı verilen teklif verme türüne alternatif olarak yeni bir teklif türü olan "average bidding"'i tanıttı. Müşterilerimizden biri olan bombabomba.com, bu yeni özelliği test etmeye karar verdi ve average bidding'in maximum bidding'den daha fazla dönüşüm getirip getirmediğini anlamak için bir A/B testi yapmak istiyor. A/B testi 1 aydır devam ediyor ve bombabomba.com şimdi sizden bu A/B testinin sonuçlarını analiz etmenizi bekliyor.

Bombabomba.com için nihai başarı ölçütü **Purchase**'dır. Bu nedenle, istatistiksel testler için Purchase metriğine odaklanılmalıdır.

## Veri Seti Hikayesi

Bir firmanın web site bilgilerini içeren bu veri setinde kullanıcıların gördükleri ve tıkladıkları reklam sayıları gibi bilgilerin yanı sıra buradan gelen kazanç bilgileri yer almaktadır. Kontrol ve Test grubu olmak üzere iki ayrı veri seti vardır. Bu veri setleri `ab_testing.xlsx` excel'inin ayrı sayfalarında yer almaktadır. Kontrol grubuna Maximum Bidding, test grubuna Average Bidding uygulanmıştır.

**Değişkenler**

- **Impression:** Reklam görüntüleme sayısı
- **Click:** Görüntülenen reklama tıklama sayısı
- **Purchase:** Tıklanan reklamlar sonrası satın alınan ürün sayısı
- **Earning:** Satın alınan ürünler sonrası elde edilen kazanç

## Proje Görevleri

1. **GÖREV 1:** Veriyi Hazırlama ve Analiz Etme
2. **GÖREV 2:** A/B Testinin Hipotezinin Tanımlanması
3. **GÖREV 3:** Hipotez Testinin Gerçekleştirilmesi
4. **GÖREV 4:** Sonuçların Analizi

## AB Testing (Bağımsız İki Örneklem T Testi)

1. Hipotezleri Kur
2. Varsayım Kontrolü
   - Normallik Varsayımı (shapiro)
   - Varyans Homojenliği (levene)
3. Hipotezin Uygulanması
   - Varsayımlar sağlanıyorsa bağımsız iki örneklem t testi
   - Varsayımlar sağlanmıyorsa mannwhitneyu testi
4. p-value değerine göre sonuçları yorumla

**Not:**
- Normallik sağlanmıyorsa direkt 2 numara. Varyans homojenliği sağlanmıyorsa 1 numaraya argüman girilir.
- Normallik incelemesi öncesi aykırı değer incelemesi ve düzeltmesi yapmak faydalı olabilir.

In [13]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import shapiro, levene, ttest_ind, mannwhitneyu

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 500)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
pd.set_option('display.max_columns', None)



---
## GÖREV 1: Veriyi Hazırlama ve Analiz Etme

### Adım 1: Veriyi Okuma

`ab_testing.xlsx` adlı kontrol ve test grubu verilerinden oluşan veri setini okutunuz. Kontrol ve test grubu verilerini ayrı değişkenlere atayınız.

In [14]:
# GÖREV 1 - ADIM 1: Veriyi okuma
df_control = pd.read_excel("datasets/ab_testing.xlsx", sheet_name="Control Group")
df_test = pd.read_excel("datasets/ab_testing.xlsx", sheet_name="Test Group")


df_control.head()

,Impression,Click,Purchase,Earning
0,82529.459,6090.077,665.211,2311.277
1,98050.452,3382.862,315.085,1742.807
2,82696.024,4167.966,458.084,1797.827
3,109914.400,4910.882,487.091,1696.229
4,108457.763,5987.656,441.034,1543.720


In [15]:
df_test.head()

,Impression,Click,Purchase,Earning
0,120103.504,3216.548,702.160,1939.611
1,134775.943,3635.082,834.054,2929.406
2,107806.621,3057.144,422.934,2526.245
3,116445.276,4650.474,429.034,2281.429
4,145082.517,5201.388,749.860,2781.698


### Adım 2: Veriyi Analiz Etme

Kontrol ve test grubu verilerini analiz ediniz.

In [16]:
# GÖREV 1 - ADIM 2: Kontrol ve test grubu verilerini analiz etme
df_test.head()

,Impression,Click,Purchase,Earning
0,120103.504,3216.548,702.160,1939.611
1,134775.943,3635.082,834.054,2929.406
2,107806.621,3057.144,422.934,2526.245
3,116445.276,4650.474,429.034,2281.429
4,145082.517,5201.388,749.860,2781.698


In [17]:
df_test.describe().T

,count,mean,std,min,25%,50%,75%,max
Impression,40.000,120512.412,18807.449,79033.835,112691.971,119291.301,132050.579,158605.920
Click,40.000,3967.550,923.095,1836.630,3376.819,3931.360,4660.498,6019.695
Purchase,40.000,582.106,161.153,311.630,444.627,551.356,699.862,889.910
Earning,40.000,2514.891,282.731,1939.611,2280.537,2544.666,2761.545,3171.490


In [18]:
df_control.describe().T

,count,mean,std,min,25%,50%,75%,max
Impression,40.000,101711.449,20302.158,45475.943,85726.690,99790.701,115212.817,147539.336
Click,40.000,5100.657,1329.985,2189.753,4124.304,5001.221,5923.804,7959.125
Purchase,40.000,550.894,134.108,267.029,470.096,531.206,637.957,801.795
Earning,40.000,1908.568,302.918,1253.990,1685.847,1975.161,2119.803,2497.295


### Adım 3: Verileri Birleştirme

Analiz işleminden sonra `concat` metodunu kullanarak kontrol ve test grubu verilerini birleştiriniz.

In [19]:
# Burada, kontrol ve test grubu veri setlerini tek bir DataFrame'de birleştiriyoruz.
# pd.concat fonksiyonu, iki ayrı DataFrame'i (df_control ve df_test) bir araya getirir.
# ignore_index=True ifadesi, eski indexleri dikkate almaz ve yeni birleşik veri seti için sıfırdan başlayarak yeniden index oluşturur.
df = pd.concat([df_control, df_test], ignore_index=True)

In [20]:
df.head()

,Impression,Click,Purchase,Earning
0,82529.459,6090.077,665.211,2311.277
1,98050.452,3382.862,315.085,1742.807
2,82696.024,4167.966,458.084,1797.827
3,109914.400,4910.882,487.091,1696.229
4,108457.763,5987.656,441.034,1543.720


In [21]:
df["Purchase"].mean()

np.float64(566.5000777093495)

---
## GÖREV 2: A/B Testinin Hipotezinin Tanımlanması

### Adım 1: Hipotezi Tanımlama

Hipotezi tanımlayınız.

**H0:** Maximum Bidding (Kontrol) ve Average Bidding (Test) gruplarının Purchase ortalamaları arasında istatistiksel olarak anlamlı bir fark yoktur. (μ_kontrol = μ_test)

**H1:** Maximum Bidding (Kontrol) ve Average Bidding (Test) gruplarının Purchase ortalamaları arasında istatistiksel olarak anlamlı bir fark vardır. (μ_kontrol ≠ μ_test)

### Adım 2: Purchase Ortalamalarını Analiz Etme

Kontrol ve test grubu için purchase ortalamalarını analiz ediniz.

In [22]:
# Kontrol ve test gruplarındaki satın alma (Purchase) ortalamalarını inceleyelim.
# Kontrol grubu: Maximum Bidding kullanan grup
control_purchase_mean = df_control["Purchase"].mean()
print(f"Kontrol Grubu Purchase Ortalaması: {control_purchase_mean}")

# Test grubu: Average Bidding kullanan grup
test_purchase_mean = df_test["Purchase"].mean()
print(f"Test Grubu Purchase Ortalaması: {test_purchase_mean}")


Kontrol Grubu Purchase Ortalaması: 550.8940587702316
Test Grubu Purchase Ortalaması: 582.1060966484677


---
## GÖREV 3: Hipotez Testinin Gerçekleştirilmesi

### AB Testing (Bağımsız İki Örneklem T Testi)

### Adım 1: Varsayım Kontrolleri

Hipotez testi yapılmadan önce varsayım kontrollerini yapınız. Bunlar Normallik Varsayımı ve Varyans Homojenliğidir.

Kontrol ve test grubunun normallik varsayımına uyup uymadığını Purchase değişkeni üzerinden ayrı ayrı test ediniz.

In [23]:
# Hipotez testi yapmadan önce varsayım kontrolleri gerçekleştirilir.
# Öncelikle, normallik varsayımını kontrol etmek için Shapiro-Wilk testi uygulanır.

test_stat, pvalue = shapiro(df_control["Purchase"])
print("Kontrol Grubu Normallik Testi Sonucu:")
print(f"Test İstatistiği = {test_stat:.4f}, p-value = {pvalue:.4f}")
# Eğer p-value 0.05'ten büyükse, normallik varsayımı sağlanmaktadır.

test_stat, pvalue = shapiro(df_test["Purchase"])
print("Test Grubu Normallik Testi Sonucu:")
print(f"Test İstatistiği = {test_stat:.4f}, p-value = {pvalue:.4f}")
# Buradaki p-value değeri de, test grubu için normallik varsayımının geçerli olup olmadığını gösterir.

# Sonrasında, varyansların homojenliğini kontrol etmek için Levene testi yapılır.
test_stat, pvalue = levene(df_control["Purchase"], df_test["Purchase"])
print("Varyans Homojenliği Testi (Levene) Sonucu:")
print(f"Test İstatistiği = {test_stat:.4f}, p-value = {pvalue:.4f}")
# Eğer p-value 0.05'ten büyükse, varyans homojenliği varsayımı sağlanmaktadır.

Kontrol Grubu Normallik Testi Sonucu:
Test İstatistiği = 0.9773, p-value = 0.5891
Test Grubu Normallik Testi Sonucu:
Test İstatistiği = 0.9589, p-value = 0.1541
Varyans Homojenliği Testi (Levene) Sonucu:
Test İstatistiği = 2.6393, p-value = 0.1083


### Adım 2: Uygun Testi Seçme

Normallik Varsayımı ve Varyans Homojenliği sonuçlarına göre uygun testi seçiniz.

In [24]:
# GÖREV 3 - ADIM 2: Burada, normallik ve varyans homojenliği varsayımları sağlandığı için bağımsız iki örneklem t testi (Student's t-test) uygulanır.
# Bu test ile kontrol ve test grubunun "Purchase" (Satın Alma) ortalamaları arasında istatistiksel olarak anlamlı bir fark olup olmadığı değerlendirilir.
# ttest_ind fonksiyonu ile iki grubun ortalamaları karşılaştırılır ve eşit varyans varsayımı (equal_var=True) kabul edilmiştir.
test_stat, pvalue = ttest_ind(df_control["Purchase"], df_test["Purchase"], equal_var=True)
print(f"Bağımsız İki Örneklem t Testi Sonucu:\nTest İstatistiği = {test_stat:.4f}, p-value = {pvalue:.4f}")
# Çıkan p-value değeri 0.05'ten büyükse, iki grup arasında "Purchase" ortalamaları açısından istatistiksel olarak anlamlı bir fark yoktur.


Bağımsız İki Örneklem t Testi Sonucu:
Test İstatistiği = -0.9416, p-value = 0.3493


### Adım 3: Sonuçları Yorumlama

Test sonucunda elde edilen p_value değerini göz önünde bulundurarak kontrol ve test grubu satın alma ortalamaları arasında istatistiki olarak anlamlı bir fark olup olmadığını yorumlayınız.

**Yorum:** p-value değeri 0.3493 olup 0.05'ten büyüktür. Bu nedenle H0 hipotezi reddedilemez. Kontrol grubu (Maximum Bidding) ve test grubu (Average Bidding) arasında Purchase ortalamaları açısından istatistiksel olarak anlamlı bir fark bulunmamaktadır.

---
## GÖREV 4: Sonuçların Analizi

### Adım 1: Kullanılan Test ve Sebepleri

Hangi testi kullandınız, sebeplerini belirtiniz.

**Kullanılan test:** Bağımsız İki Örneklem t Testi (Independent Samples t-Test)

**Sebep:** Her iki grup için Shapiro testi sonucunda normallik varsayımı sağlanmaktadır (Kontrol p=0.5891, Test p=0.1541). Levene testi sonucunda varyans homojenliği varsayımı da sağlanmaktadır (p=0.1083). Bu nedenle varsayımlar karşılandığı için bağımsız iki örneklem t testi uygulanmıştır.

### Adım 2: Müşteriye Tavsiye

Elde ettiğiniz test sonuçlarına göre müşteriye tavsiyede bulununuz.

**Tavsiye:** Average Bidding yönteminin Maximum Bidding'e kıyasla Purchase metriğinde istatistiksel olarak anlamlı bir artış sağlamadığı görülmektedir (Kontrol ort.: 550.89, Test ort.: 582.11, p=0.3493). Bu nedenle bombabomba.com'un mevcut Maximum Bidding stratejisine devam etmesi önerilir. Dilerseniz daha uzun süreli veya daha büyük örneklemli bir test ile sonuçları tekrar doğrulayabilirsiniz.